# Ver lo que aprende una representación — cuaderno de exploración

Una imagen puede perder píxeles sin perder su estructura. ¿Aprender a recuperar
lo que falta produce una representación útil con pocas etiquetas?

Las imágenes y los puntos son **sintéticos**, creados aquí (CC0-1.0). Este
experimento usa CPU y no descarga nada. Entrena redes pequeñas desde cero; no
reproduce un encoder preentrenado ni una tarea IOAI. Las figuras son resultados
de tu ejecución. Cambia una condición y mira qué explicación sobrevive.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(23)
rng = np.random.default_rng(23)
plt.rcParams.update({'figure.figsize': (10, 3), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})


In [ ]:
ocultar = 0.4
epocas = 65
pasos_gan = 450


In [ ]:
def imagenes(n, ruido=0.14, semilla=0):
    r = np.random.default_rng(semilla)
    yy, xx = np.mgrid[:16, :16]
    etiquetas = r.integers(0, 2, n)
    imagenes = []
    for clase in etiquetas:
        centro = r.uniform(4, 11)
        coordenada = yy if clase == 0 else xx
        barra = np.exp(-((coordenada - centro) / 1.5) ** 2)
        imagenes.append(np.clip(barra + r.normal(0, ruido, (16, 16)), 0, 1))
    return np.asarray(imagenes, dtype=np.float32), etiquetas

train, y_train = imagenes(384, semilla=0)
val, y_val = imagenes(192, semilla=1)
transfer, y_transfer = imagenes(192, ruido=.32, semilla=2)
X = torch.from_numpy(train.reshape(-1, 256))
V = torch.from_numpy(val.reshape(-1, 256))
T = torch.from_numpy(transfer.reshape(-1, 256))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, im, y in zip(axes, train, y_train):
    ax.imshow(im, cmap='magma', vmin=0, vmax=1)
    ax.set_title('horizontal' if y == 0 else 'vertical', fontsize=9)
    ax.axis('off')
plt.show()


## Aprender sin las etiquetas

Ocultamos píxeles y pedimos reconstruir la imagen completa. Un vector de ocho
números debe transportar lo aprendido. La referencia trivial es la imagen media
de train; la comparación usa imágenes nuevas y una máscara fija.


In [ ]:
# Un autoencoder aprende a reconstruir lo que ocultamos de cada imagen.
# Las etiquetas de orientación no participan en este entrenamiento.
encoder = nn.Sequential(nn.Linear(256, 48), nn.ReLU(), nn.Linear(48, 8))
decoder = nn.Sequential(nn.Linear(8, 48), nn.ReLU(), nn.Linear(48, 256), nn.Sigmoid())
autoencoder = nn.Sequential(encoder, decoder)
opt = torch.optim.Adam(autoencoder.parameters(), lr=.004)
curva = []
for epoca in range(epocas):
    perdidas = []
    for indices in torch.randperm(len(X)).split(64):
        limpio = X[indices]
        visible = (torch.rand_like(limpio) > ocultar).float()
        reconstruido = autoencoder(limpio * visible)
        perdida = F.mse_loss(reconstruido, limpio)
        opt.zero_grad(); perdida.backward(); opt.step()
        perdidas.append(perdida.item())
    curva.append(float(np.mean(perdidas)))

autoencoder.eval()
# Máscaras fijas para que la comparación no cambie al redibujar la figura.
g = torch.Generator().manual_seed(91)
mascara_val = (torch.rand(V.shape, generator=g) > ocultar).float()
mascara_transfer = (torch.rand(T.shape, generator=g) > ocultar).float()
with torch.inference_mode():
    reconstruido = autoencoder(V * mascara_val)
    reconstruido_transfer = autoencoder(T * mascara_transfer)
    media_train = X.mean(0).expand_as(V)
    base_mse = F.mse_loss(media_train, V).item()
    val_mse = F.mse_loss(reconstruido, V).item()
    transfer_mse = F.mse_loss(reconstruido_transfer, T).item()

# Guardas del material: el caso de referencia debe aprender, no solo ejecutar.
assert reconstruido.shape == V.shape
assert torch.isfinite(reconstruido).all() and torch.isfinite(reconstruido_transfer).all()
if epocas == 65 and ocultar in (.4, .6):
    assert val_mse < base_mse * .5, 'La reconstrucción no aprendió el caso de referencia'
    assert curva[-1] < curva[0] * .5, 'La pérdida no descendió en el caso de referencia'

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for col in range(6):
    for fila, datos in enumerate((V, V * mascara_val, reconstruido)):
        axes[fila, col].imshow(datos[col].reshape(16, 16), cmap='magma', vmin=0, vmax=1)
        axes[fila, col].set_xticks([]); axes[fila, col].set_yticks([])
for ax, titulo in zip(axes[:, 0], ['Original', 'Entrada incompleta', 'Reconstrucción']):
    ax.set_ylabel(titulo)
fig.tight_layout(); plt.show()
print({'MSE imagen media de train': round(base_mse, 4),
       'MSE reconstrucción': round(val_mse, 4),
       'MSE con más ruido': round(transfer_mse, 4)})
plt.plot(curva); plt.xlabel('Época'); plt.ylabel('MSE de reconstrucción'); plt.show()


## ¿Reconstruir es lo mismo que entender?

Damos al clasificador solo 24 etiquetas. Compara píxeles y representación con el
mismo split. No hay una garantía de que la representación gane: la geometría de
los datos, el objetivo y el ruido importan. Cambia el ancho del cuello y observa
si mejora reconstrucción, clasificación, ambas o ninguna.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

with torch.inference_mode():
    z_train = encoder(X).numpy()
    z_val = encoder(V).numpy()
    z_transfer = encoder(T).numpy()
# Etiquetamos solo 24 imágenes. Son un subconjunto fijo de train.
etiquetados = np.r_[np.flatnonzero(y_train == 0)[:12], np.flatnonzero(y_train == 1)[:12]]
sondas = {}
for nombre, a, b, c in [
    ('Píxeles', X.numpy(), V.numpy(), T.numpy()),
    ('Representación aprendida', z_train, z_val, z_transfer),
]:
    sonda = make_pipeline(StandardScaler(), LogisticRegression(C=.1, max_iter=1000, random_state=0))
    sonda.fit(a[etiquetados], y_train[etiquetados])
    sondas[nombre] = [accuracy_score(y_val, sonda.predict(b)),
                     accuracy_score(y_transfer, sonda.predict(c))]
fig, ax = plt.subplots(figsize=(8, 3))
x = np.arange(len(sondas))
ax.bar(x-.15, [v[0] for v in sondas.values()], width=.3, label='Validación')
ax.bar(x+.15, [v[1] for v in sondas.values()], width=.3, label='Más ruido')
ax.set_xticks(x, sondas.keys()); ax.set_ylim(0, 1.05); ax.set_ylabel('Accuracy')
ax.legend(); plt.show()


## Un generador que tiene que inventar

Una GAN enfrenta dos redes: G produce puntos y D distingue reales de generados.
Los cuatro grupos son fáciles de ver; por eso aquí se puede detectar un colapso
que una pérdida promedio escondería. Se mantienen los mismos latentes antes y
después. El entrenamiento puede cubrir los cuatro modos o concentrarse en unos
pocos: ese comportamiento es parte del experimento.


In [ ]:
# El segundo experimento es una GAN diminuta sobre puntos 2D, no un modelo de imágenes.
def reales(n):
    centros = torch.tensor([[-1., -1.], [-1., 1.], [1., -1.], [1., 1.]])
    return centros[torch.randint(4, (n,))] + .18 * torch.randn(n, 2)

G = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 2))
D = nn.Sequential(nn.Linear(2, 32), nn.LeakyReLU(.2), nn.Linear(32, 32), nn.LeakyReLU(.2), nn.Linear(32, 1))
opt_g = torch.optim.Adam(G.parameters(), lr=.001, betas=(.5, .999))
opt_d = torch.optim.Adam(D.parameters(), lr=.001, betas=(.5, .999))
z_fijo = torch.randn(512, 2)
with torch.inference_mode(): antes = G(z_fijo).numpy()

def copiar_pesos(red):
    return [p.detach().clone() for p in red.parameters()]

def mismos_pesos(red, copia):
    return all(torch.equal(p.detach(), anterior) for p, anterior in zip(red.parameters(), copia))

for paso in range(pasos_gan):
    g_antes_d = copiar_pesos(G)
    d_antes_d = copiar_pesos(D)
    for p in D.parameters(): p.requires_grad_(True)
    r = reales(128); falso = G(torch.randn(128, 2)).detach()
    loss_d = F.binary_cross_entropy_with_logits(D(r), torch.ones(128, 1)) + F.binary_cross_entropy_with_logits(D(falso), torch.zeros(128, 1))
    opt_d.zero_grad(); loss_d.backward(); opt_d.step()
    assert mismos_pesos(G, g_antes_d), 'El paso de D modificó G'
    if paso == 0:
        assert not mismos_pesos(D, d_antes_d), 'El primer paso de D no aprendió'
    d_antes_g = copiar_pesos(D)
    # D deja pasar el gradiente hacia G, pero sus pesos no se actualizan.
    for p in D.parameters(): p.requires_grad_(False)
    loss_g = F.binary_cross_entropy_with_logits(D(G(torch.randn(128, 2))), torch.ones(128, 1))
    opt_g.zero_grad(); loss_g.backward(); opt_g.step()
    assert mismos_pesos(D, d_antes_g), 'El paso de G modificó D'
    if paso == 0:
        assert not mismos_pesos(G, g_antes_d), 'El primer paso de G no aprendió'
    assert torch.isfinite(loss_d) and torch.isfinite(loss_g)
with torch.inference_mode(): despues = G(z_fijo).numpy()
assert np.isfinite(antes).all() and np.isfinite(despues).all()
if pasos_gan > 0:
    assert not np.array_equal(antes, despues), 'El generador no cambió sus salidas'
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, datos, titulo in zip(axes, (reales(512).numpy(), antes, despues), ('Distribución real', 'G antes', 'G después')):
    ax.scatter(*datos.T, s=6, alpha=.4)
    ax.set(xlim=(-2, 2), ylim=(-2, 2), title=titulo, aspect='equal')
plt.show()
# Diversidad: cuentas por cuadrante. Una pérdida pequeña no garantiza cuatro modos.
cuadrante = (despues[:, 0] > 0).astype(int) + 2 * (despues[:, 1] > 0).astype(int)
print('Muestras por cuadrante:', np.bincount(cuadrante, minlength=4).tolist())


¿Puedes reducir la pérdida del autoencoder y empeorar la clasificación?
¿Puedes obtener puntos plausibles con la GAN que cubran solo un grupo? Cambia el
ruido, el cuello o la tasa de aprendizaje y compara las figuras; conserva una
sola condición distinta para entender qué produjo el cambio.


El archivo siguiente es un resumen técnico para verificar el cuaderno automáticamente; no hay un formulario que completar.


In [ ]:
resultado = {'laboratorio': '08_vision', 'version': 'cuaderno de exploración',
             'metrica': 'MSE de reconstrucción, menor es mejor', 'split': '384 imágenes train, 192 validación independiente, 192 transferencia con más ruido',
             'baseline': base_mse, 'validacion': val_mse, 'transferencia': transfer_mse}
assert all(np.isfinite(resultado[k]) for k in ('baseline', 'validacion', 'transferencia'))
Path('resultado.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=2) + '\n')
